In [ ]:
import copy
import itertools
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.nn.functional import relu, sigmoid
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [ ]:
np.random.seed(110007)

In [ ]:
transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])

train_dataset = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root="data", train=False, download=True, transform=transform)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 340)
        self.fc2 = nn.Linear(340, 150)
        self.fc3 = nn.Linear(150, 340)
        self.fc4 = nn.Linear(340, 28 * 28)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = relu(self.fc1(x))
        x = relu(self.fc2(x))
        x = relu(self.fc3(x))
        x = sigmoid(self.fc4(x))
        return x

In [ ]:
def train_autoencoder(model, epochs=5):

    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.BCELoss()
    
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Model size: {total_params}')
    
    train_losses = []
    
    for e in range(epochs):
        running_loss = 0.0
        for i, data in enumerate(train_loader, 0):
            X, _ = data
            optimizer.zero_grad()
            X_hat = model(X)
            loss = criterion(X_hat, torch.flatten(X, 1))
            loss.backward()
            optimizer.step()
    
            running_loss += loss.item()
    
        epoch_loss = running_loss / len(train_dataset)
        train_losses.append(epoch_loss)
        
        print(f'[Epoch {e+1}] Loss: {epoch_loss:.4f}')

def test_autoencoder(model):
    loss = 0.0 
    criterion = nn.BCELoss()
    with torch.no_grad():
        for data in test_loader:
            X, _ = data
            X_hat = model(X)
            loss += criterion(X_hat, torch.flatten(X, 1)).item()
    loss /= len(test_dataset)
    print(f'Loss of the network on the test images: {loss:.4f}')

In [ ]:
autoencoder = Autoencoder()
train_autoencoder(autoencoder)

In [ ]:
test_autoencoder(autoencoder)

In [ ]:
class AEClassifier(nn.Module):
    def __init__(self, autoencoder):
        super().__init__()
        self.fc1 = copy.deepcopy(autoencoder.fc1)
        self.fc2 = copy.deepcopy(autoencoder.fc2)
        for param in self.fc1.parameters():
            param.requires_grad = False
        for param in self.fc2.parameters():
            param.requires_grad = False
        self.fc3 = nn.Linear(150, 10)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = relu(self.fc1(x))
        x = relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(28 * 28, 10)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [ ]:
def train(model, epochs=5):

    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Model size: {total_params}')
    
    train_losses = []
    
    for e in range(epochs):
        running_loss = 0.0
        for i, data in enumerate(train_loader, 0):
            X, y = data
            optimizer.zero_grad()
            y_hat = model(X)
            loss = criterion(y_hat, y)
            loss.backward()
            optimizer.step()
    
            running_loss += loss.item()
    
        epoch_loss = running_loss / len(train_dataset)
        train_losses.append(epoch_loss)
        
        print(f'[Epoch {e+1}] Loss: {epoch_loss:.4f}')

def test(model):
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data in test_loader:
            X, y = data
            output = model(X)
            _, y_hat = torch.max(output.data, 1)
            total += y.size(0)
            correct += (y_hat == y).sum().item()
    
    print(f'\nAccuracy of the network on the {total} test images: {100 * correct / total} %')

In [ ]:
ae_classifier = AEClassifier(autoencoder)

start = time.time()
train(ae_classifier)
end = time.time()
print(f'Training took: {end - start:.2f} sec')

test(ae_classifier)

In [ ]:
mlp_classifier = MLPClassifier()

start = time.time()
train(mlp_classifier)
end = time.time()
print(f'Training took: {end - start:.2f} sec')

test(mlp_classifier)